# 07. Final Results Summary

## Objective

This notebook consolidates the final results of the TCGA-KIRC
patient-level PFI prognosis project.

The analyses and models were completed in the previous notebooks.
No model selection or hyperparameter tuning is performed here.

This notebook summarizes:

1. Cohort and Train / Validation / Test split
2. Performance of the four frozen survival models
   - RNA-only
   - Clinical-only
   - Clinical + RNA
   - Clinical + Hallmark
3. Incremental value of RNA over the Clinical-only model
4. Clinical + RNA model interpretation
5. Molecular Twin and Molecular Support results
6. Patient-level inference and reproducibility QC
7. Final conclusions and limitations

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

# Project paths
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

print("Project root:", PROJECT_ROOT)

Project root: c:\kirc-genomic-prognosis


In [2]:
# Load final modeling cohort
cohort = pd.read_parquet(
    PROCESSED_DIR / "kirc_modeling_cohort.parquet"
)

print("Modeling cohort:", cohort.shape)
print("\nPFI event distribution:")
print(cohort["pfi_event"].value_counts().sort_index())

print("\nPFI event rate:")
print(cohort["pfi_event"].mean())

Modeling cohort: (529, 10)

PFI event distribution:
pfi_event
0.0    370
1.0    159
Name: count, dtype: int64

PFI event rate:
0.3005671077504726


In [3]:
from sklearn.model_selection import train_test_split

SEED = 42

# Reproduce the frozen event-stratified split
train_df, temp_df = train_test_split(
    cohort,
    test_size=0.30,
    random_state=SEED,
    stratify=cohort["pfi_event"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=80 / 159,
    random_state=SEED,
    stratify=temp_df["pfi_event"]
)

split_summary = pd.DataFrame({
    "Cohort": ["Train", "Validation", "Test"],
    "N": [len(train_df), len(val_df), len(test_df)],
    "Events": [
        int(train_df["pfi_event"].sum()),
        int(val_df["pfi_event"].sum()),
        int(test_df["pfi_event"].sum())
    ],
})

split_summary["Event rate"] = (
    split_summary["Events"] / split_summary["N"]
)

split_summary

,Cohort,N,Events,Event rate
0,Train,370,111,0.300000
1,Validation,79,24,0.303797
2,Test,80,24,0.300000


In [4]:
brier_df = pd.read_csv(
    PROCESSED_DIR / "test_brier_ibs_metrics.csv"
)

brier_df

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\kirc-genomic-prognosis\\data\\processed\\test_brier_ibs_metrics.csv'

In [5]:
print("Processed files:")

for f in sorted(PROCESSED_DIR.iterdir()):
    print(f.name)

Processed files:
kirc_modeling_cohort.parquet
rna_tpm_modeling_raw.parquet
splits.csv


In [6]:
final_performance = pd.DataFrame({
    "Model": [
        "RNA-only",
        "Clinical-only",
        "Clinical+RNA",
        "Clinical+Hallmark"
    ],
    "Test Harrell C-index": [
        0.715,
        0.887,
        0.892,
        0.886
    ],
    "1-year AUC": [
        0.708,
        0.888,
        0.880,
        0.901
    ],
    "3-year AUC": [
        0.723,
        0.911,
        0.944,
        0.895
    ],
    "IBS (1–3 years)": [
        0.148,
        0.092,
        0.099,
        0.093
    ]
})

final_performance

,Model,Test Harrell C-index,1-year AUC,3-year AUC,IBS (1–3 years)
0,RNA-only,0.715,0.708,0.723,0.148
1,Clinical-only,0.887,0.888,0.911,0.092
2,Clinical+RNA,0.892,0.880,0.944,0.099
3,Clinical+Hallmark,0.886,0.901,0.895,0.093


## Incremental Value of RNA

The Clinical+RNA model achieved a slightly higher test Harrell C-index
than the Clinical-only model (0.892 vs. 0.887).

However, paired bootstrap analysis showed:

- ΔC (Clinical+RNA − Clinical-only): **+0.0051**
- 95% bootstrap CI: **−0.0243 to +0.0440**

Because the confidence interval includes zero, the test-set analysis
does not provide clear evidence that adding RNA features improves
discrimination over the Clinical-only model.

Therefore, the RNA component should not be interpreted as providing
a statistically clear incremental discrimination benefit in this cohort.

## Clinical + RNA Model Interpretation

The Clinical+RNA CoxNet model used **510 input features**:

- 10 encoded clinical features
- 500 RNA expression features selected using the training set

At the selected regularization parameter, **24 features had non-zero
coefficients**:

- 3 clinical features
- 21 RNA features

### Global interpretation

CoxNet coefficients were used to examine the direction and magnitude
of feature contributions to the model's linear risk score.

### Patient-level interpretation

For each patient, the local contribution of feature \(j\) was calculated as:

\[
\text{Contribution}_j = x_j \times \beta_j
\]

where \(x_j\) is the patient's transformed and standardized feature value
and \(\beta_j\) is the corresponding CoxNet coefficient.

The sum of all feature contributions reproduces the patient's Cox model
risk score (linear predictor) to numerical precision.

These contributions describe how features influence the model's risk score.
They should not be interpreted as causal effects or as absolute changes in
recurrence probability.

## Molecular Twin and Molecular Support

### Molecular Twin

For each Test patient, the five most similar patients in the Train cohort
were identified using RNA-expression similarity.

The similarity search used only molecular features and did not use clinical
variables, PFI event status, or PFI time.

Molecular Twin preprocessing:

- Protein-coding genes: 19,962
- Train-derived low-expression filtering: 14,220 genes retained
- log2(TPM + 1) transformation
- Top 500 genes selected by MAD using the Train cohort
- StandardScaler fitted on the Train cohort
- Cosine similarity
- Top-5 Train patients retrieved for each Test patient

Across the 80 Test patients:

- 400 patient–twin pairs were generated
- 202 unique Train patients appeared as twins
- Mean similarity: 0.5083
- Median similarity: 0.4939
- Similarity range: 0.2374–0.9636

PFI outcomes of the retrieved Train patients were appended only after
similarity search for retrospective context and were not used to identify
molecular twins.

### Molecular Support

Molecular Support was defined using each Train patient's nearest-neighbor
similarity to another Train patient.

The support threshold was the **5th percentile of Train nearest-neighbor
similarity**:

**Threshold = 0.3810**

A Test patient was flagged as having low molecular support when:

\[
\text{maximum similarity to Train} < 0.3810
\]

Using this Train-derived threshold:

- 4 of 80 Test patients were flagged as low support
- Low-support rate: 5%

This flag represents low similarity to the training cohort and should not
be interpreted as formal proof that a patient is out-of-distribution (OOD).

## Patient-level Inference Pipeline

The final patient-level pipeline integrates the frozen Clinical+RNA CoxNet
model with model interpretation and molecular-reference information.

The pipeline produces:

1. Clinical+RNA CoxNet risk score
2. Patient-level feature contributions
3. Top-5 Molecular Twins
4. Molecular Support assessment
5. Integrated patient report

The CoxNet output is a relative risk score (linear predictor) and should
not be interpreted as an absolute probability of recurrence or progression.

Molecular Twin outcomes provide retrospective examples from similar
training patients and are not predictions of the query patient's outcome.

### Reproducibility QC

The reconstructed inference pipeline was evaluated for all **80 Test
patients**.

Results:

- Patients checked: **80**
- Maximum difference between batch and single-patient risk scores:
  **4.44 × 10⁻¹⁶**
- Maximum difference between model risk score and the sum of local XAI
  contributions: **4.44 × 10⁻¹⁶**
- Batch vs. single-patient risk agreement: **80 / 80**
- Risk score vs. XAI contribution-sum agreement: **80 / 80**

These differences are within floating-point numerical precision, confirming
internal consistency of the reconstructed patient-level inference pipeline.

## Final Conclusions

This project developed and evaluated patient-level PFI prognostic models
for TCGA-KIRC using clinical and RNA-seq information.

### Main findings

- The final modeling cohort included **529 patients**, with an event rate
  of approximately **30.1%**.
- The Clinical+RNA model achieved a Test Harrell C-index of **0.892**,
  compared with **0.887** for the Clinical-only model.
- Clinical+RNA showed the highest 3-year time-dependent AUC (**0.944**),
  while Clinical+Hallmark showed the highest 1-year AUC (**0.901**).
- Clinical-only achieved the lowest 1–3 year IBS (**0.092**).
- The paired bootstrap difference between Clinical+RNA and Clinical-only
  was small (ΔC = **+0.0051**, 95% CI **−0.0243 to +0.0440**).

Therefore, although molecular information contributed to a richer
patient-level interpretation framework, this test-set analysis did not
show clear evidence of an incremental discrimination benefit from adding
RNA features to the Clinical-only model.

The final pipeline additionally provides patient-level model
interpretation, molecularly similar training patients, and an indication
of molecular support relative to the training cohort.

## Limitations

1. The analysis was performed using a single TCGA-KIRC cohort and lacks
   independent external validation.

2. The Test cohort contained only **80 patients and 24 PFI events**,
   resulting in uncertainty in performance estimates and model comparisons.

3. Feature selection, model development, and evaluation were performed
   within this retrospective dataset. Generalizability to other clinical
   populations has not been established.

4. CoxNet risk scores represent relative prognostic scores and are not
   calibrated absolute probabilities of progression or recurrence.

5. Model coefficients and patient-level feature contributions describe
   associations within the fitted prediction model and should not be
   interpreted as causal biological effects.

6. Molecular Twins are based on RNA-expression similarity within the
   training cohort. Outcomes of similar patients are retrospective
   references, not predictions of an individual patient's outcome.

7. Molecular Support is a training-cohort similarity heuristic and is
   not a formal out-of-distribution detection method.